# Apply best Heretic trial & upload

The 100-trial Optuna search is **already done** — the checkpoint JSONL has the results.
This notebook loads it, picks the best trial, applies the abliteration, and uploads
the merged model to Hugging Face. ~10 min on a T4.

### Setup
1. **+ Add Input → Upload → New Dataset** — upload `Qwen--Qwen3-4B-Thinking-2507.jsonl`
2. **Settings → Accelerator → GPU**, **Internet → On**
3. Replace `hf_YOUR_TOKEN_HERE` below with your real HF write token
4. Run All

In [ ]:
# ── Config ──
MODEL_ID     = "Qwen/Qwen3-4B-Thinking-2507"
MODEL_COMMIT = "768f209d9ea81521153ed38c47d515654e938aea"
REPO_ID      = "d0gra/Qwen3-4B-Thinking-2507-say-no-more"
HF_TOKEN     = "hf_YOUR_TOKEN_HERE"
N_TRIALS     = "100"
SEED         = "0"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU"
%pip install -q "numpy>=2.2,<2.3" heretic-llm
%pip uninstall -y torchao 2>/dev/null || true

In [ ]:
import os, glob, shutil

# Find the uploaded checkpoint
src = glob.glob("/kaggle/input/**/Qwen--Qwen3-4B-Thinking-2507.jsonl", recursive=True)
assert src, (
    "Checkpoint not found! Upload Qwen--Qwen3-4B-Thinking-2507.jsonl "
    "via + Add Input → Upload → New Dataset"
)
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
shutil.copy(src[0], "/kaggle/working/checkpoints/Qwen--Qwen3-4B-Thinking-2507.jsonl")
print("Checkpoint ready:", os.listdir("/kaggle/working/checkpoints"))

In [ ]:
%%writefile /kaggle/working/run_apply.py
import os, sys, huggingface_hub

MODEL_ID     = os.environ["MODEL_ID"]
MODEL_COMMIT = os.environ.get("MODEL_COMMIT") or None
REPO_ID      = os.environ["REPO_ID"]
N_TRIALS     = os.environ.get("N_TRIALS", "100")
SEED         = os.environ.get("SEED", "0")
CHECKPOINT   = os.environ.get("CHECKPOINT", "/kaggle/working/checkpoints")
HF_TOKEN     = os.environ["HF_TOKEN"]

huggingface_hub.login(token=HF_TOKEN, add_to_git_credential=False)
huggingface_hub.get_token = lambda: HF_TOKEN

import heretic.main as hmain
import heretic.utils as hutils

_counts = {}
def _disp(c):
    if isinstance(c, str):
        return c
    t = getattr(c, "title", None)
    return t if isinstance(t, str) else str(getattr(c, "value", c))
def _val(c):
    if isinstance(c, str):
        return c
    return getattr(c, "value", c)
def _find(choices, needle):
    for c in choices:
        if needle in _disp(c).lower():
            return _val(c)
    return None

def patched_select(message, choices):
    m = str(message).lower()
    _counts[m] = _counts.get(m, 0) + 1
    labels = [_disp(c) for c in choices]
    print(f"  [auto] {message}")
    print(f"         choices: {labels}")
    if "how would you like to proceed" in m:
        pick = (_find(choices, "show the results") or _find(choices, "continue")
                or _val(choices[0]))
        print(f"         -> show the results")
        return pick
    if "which trial" in m:
        if _counts[m] == 1:
            print(f"         -> best trial (first)")
            return _val(choices[0])
        print(f"         -> exit")
        return _find(choices, "exit") or _val(choices[-1])
    if "what do you want to do" in m:
        pick = _find(choices, "upload")
        print(f"         -> upload")
        return pick
    if "export" in m:
        pick = _find(choices, "merge")
        print(f"         -> merge")
        return pick
    if "public or private" in m:
        print(f"         -> public")
        return _find(choices, "public")
    if "reproducibility" in m:
        pick = _find(choices, "full")
        print(f"         -> full")
        return pick
    print(f"         -> {labels[0]}")
    return _val(choices[0])

def patched_text(message, default="", qmark="?", unsafe=False):
    print(f"  [auto] {message}")
    if "repository" in str(message).lower():
        print(f"         -> {REPO_ID}")
        return REPO_ID
    print(f"         -> {default}")
    return default

def patched_path(message):
    print(f"  [auto] {message} -> /kaggle/working/out")
    return "/kaggle/working/out"

def patched_password(message):
    return HF_TOKEN

for mod in (hmain, hutils):
    mod.prompt_select   = patched_select
    mod.prompt_text     = patched_text
    mod.prompt_path     = patched_path
    mod.prompt_password = patched_password

sys.argv = ["heretic", "--model", MODEL_ID,
            "--export-strategy", "merge", "--n-trials", N_TRIALS, "--seed", SEED,
            "--study-checkpoint-dir", CHECKPOINT]
if MODEL_COMMIT:
    sys.argv += ["--model-commit", MODEL_COMMIT]

hmain.run()
print("\nDONE: uploaded to https://huggingface.co/" + REPO_ID)

In [ ]:
import subprocess, sys
env = {**os.environ,
       "MODEL_ID": MODEL_ID, "MODEL_COMMIT": MODEL_COMMIT,
       "REPO_ID": REPO_ID, "N_TRIALS": N_TRIALS, "SEED": SEED,
       "CHECKPOINT": "/kaggle/working/checkpoints", "HF_TOKEN": HF_TOKEN}
proc = subprocess.run([sys.executable, "/kaggle/working/run_apply.py"],
                      env=env, cwd="/kaggle/working")
if proc.returncode != 0:
    raise SystemExit(f"Heretic exited with code {proc.returncode}")

In [ ]:
from huggingface_hub import list_repo_files
print("Uploaded files:")
for f in sorted(list_repo_files(REPO_ID, token=HF_TOKEN)):
    print(" ", f)
print(f"\nModel: https://huggingface.co/{REPO_ID}")
print(f"\nOn the Mac:")
print(f"  .venv/bin/hf download {REPO_ID} --local-dir models/Qwen3-4B-Thinking-2507-say-no-more")
print(f"  .venv/bin/mlx_lm.convert --hf-path models/Qwen3-4B-Thinking-2507-say-no-more \\")
print(f"      --mlx-path models/Qwen3-4B-Thinking-2507-say-no-more-mlx-8bit -q --q-bits 8")
print(f"  MODEL=models/Qwen3-4B-Thinking-2507-say-no-more-mlx-8bit scripts/serve.sh")
print(f"  scripts/eval.sh 03-abliterated")